# Load Model

In [2]:
from transformers import AutoModel, AdamW
from transformers import get_scheduler
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
import torch.nn as nn
import pandas as pd
import random
from transformers import AutoTokenizer
from utils.load import LoadDataCSV
from utils.embeddings import Embeddings

# Load datasets

df_loader=LoadDataCSV()

from transformers import AutoModel
import torch
import torch.nn as nn
import torch.nn.functional as F

class E5FineTuner(nn.Module):
    def __init__(self, model_name, embedding_dim=128):
        """
        Fine-tuneable model for contrastive learning based on pre-trained models.
        Args:
            model_name (str): Name of the pre-trained model from Hugging Face.
            embedding_dim (int): Dimensionality of the projected embedding space.
        """
        super().__init__()
        self.model = AutoModel.from_pretrained(model_name)
        self.projection = nn.Linear(self.model.config.hidden_size, embedding_dim)  # Projection layer
        self.normalize = nn.LayerNorm(embedding_dim)  # Optional: Normalization for stability

    def forward(self, input_ids, attention_mask):
        """
        Forward pass for encoding input text.
        Args:
            input_ids (torch.Tensor): Tokenized input IDs.
            attention_mask (torch.Tensor): Attention masks for the input IDs.
        Returns:
            torch.Tensor: Normalized embeddings of shape (batch_size, embedding_dim).
        """
        # Get outputs from the transformer
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
        
        # Use pooler_output if available, else mean-pooling over last hidden state
        if hasattr(outputs, 'pooler_output') and outputs.pooler_output is not None:
            embeddings = outputs.pooler_output
        else:
            # Mean-pooling
            embeddings = torch.mean(outputs.last_hidden_state, dim=1)
        
        # Apply projection and normalization
        embeddings = self.projection(embeddings)
        embeddings = self.normalize(embeddings)
        embeddings = F.normalize(embeddings, p=2, dim=1)  # L2-normalization for cosine similarity
        
        return embeddings

tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-small")

In [3]:
import torch

# Ruta al archivo guardado
checkpoint_path = "checkpoints/run7/model_step_5000.pth"

# Cargar checkpoint
checkpoint = torch.load(checkpoint_path)

# Crear una nueva instancia del modelo y cargar el estado
model_loaded = E5FineTuner("intfloat/multilingual-e5-small")  # Usa la misma configuración que el modelo entrenado
#model_loaded.load_state_dict(checkpoint['model_state_dict'])

# Cargar el optimizador si planeas continuar el entrenamiento
#optimizer = AdamW(model_loaded.parameters(), lr=1e-5)
#optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Cargar el scheduler si lo usaste
#scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=0)  # Configuración ajustada
#scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

# Opcional: verifica la pérdida o la época desde donde cargar
#epoch = checkpoint['epoch']
#loss = checkpoint['loss']

#print(f"Modelo cargado desde la época {epoch} con pérdida {loss:.4f}")

# Configurar el dispositivo
device = "mps" if torch.backends.mps.is_available() else "cpu"
model_loaded.to(device)

# Verificar que el modelo funcione correctamente
model_loaded.eval()  # Configura el modelo para inferencia
sample_text = "This is a sample text to verify embeddings."
inputs = tokenizer(sample_text, return_tensors="pt", padding=True, truncation=True).to(device)

# Generar embeddings como prueba
with torch.no_grad():
    embeddings = model_loaded(inputs["input_ids"], inputs["attention_mask"])
    print("Embeddings generados correctamente.")
    print(embeddings.shape)


/var/folders/dj/58tclkkj76sf6w2h7x_9cyt40000gn/T/ipykernel_1086/3819800583.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Embeddings generados correctamente.
torch.Size([1, 128])


In [4]:
print(checkpoint.keys())

odict_keys(['model.embeddings.word_embeddings.weight', 'model.embeddings.position_embeddings.weight', 'model.embeddings.token_type_embeddings.weight', 'model.embeddings.LayerNorm.weight', 'model.embeddings.LayerNorm.bias', 'model.encoder.layer.0.attention.self.query.weight', 'model.encoder.layer.0.attention.self.query.bias', 'model.encoder.layer.0.attention.self.key.weight', 'model.encoder.layer.0.attention.self.key.bias', 'model.encoder.layer.0.attention.self.value.weight', 'model.encoder.layer.0.attention.self.value.bias', 'model.encoder.layer.0.attention.output.dense.weight', 'model.encoder.layer.0.attention.output.dense.bias', 'model.encoder.layer.0.attention.output.LayerNorm.weight', 'model.encoder.layer.0.attention.output.LayerNorm.bias', 'model.encoder.layer.0.intermediate.dense.weight', 'model.encoder.layer.0.intermediate.dense.bias', 'model.encoder.layer.0.output.dense.weight', 'model.encoder.layer.0.output.dense.bias', 'model.encoder.layer.0.output.LayerNorm.weight', 'model.e

# Load Data

In [5]:
fact_checks_mono_sample_1000=pd.read_csv('data/fact_checks_mono_sample_1000.csv')
posts_mono_sample_1000=pd.read_csv('data/posts_train_mono_sample_1000.csv')

embeddings_utils=Embeddings

embedding_columns_posts = ['ocr_text_embedding', 'ocr_text_eng_embedding']
embedding_columns_fact_checks = ['claim_title_eng_embedding', 'claim_title_embedding']

fact_checks_mono_sample_1000=embeddings_utils.restore_embeddings(fact_checks_mono_sample_1000, embedding_columns_fact_checks)
posts_mono_sample_1000=embeddings_utils.restore_embeddings(posts_mono_sample_1000, embedding_columns_posts)

# Load your data

df_fact_checks, df_posts, df_fact_check_post_mapping = df_loader.load_data()

# Create a df of mapping for post_id that show up in posts_mono_sample_1000
mapping_sample_1000=df_fact_check_post_mapping[df_fact_check_post_mapping['post_id'].isin(posts_mono_sample_1000['post_id'])]
# Generate extremely small samples
posts_mono_sample_1000_10=posts_mono_sample_1000.sample(n=10)

# Create a df of mapping for post_id that show up in posts_mono_sample_1000
mapping_sample_1000_10=df_fact_check_post_mapping[df_fact_check_post_mapping['post_id'].isin(posts_mono_sample_1000_10['post_id'])]

fact_checks_mono_sample_1000_10=fact_checks_mono_sample_1000[fact_checks_mono_sample_1000['fact_check_id'].isin(mapping_sample_1000_10['fact_check_id'])]

In [6]:
posts_mono_sample_1000 = posts_mono_sample_1000.set_index("post_id")
fact_checks_mono_sample_1000 = fact_checks_mono_sample_1000.set_index("fact_check_id")

# Generate Embeddings

In [7]:
import torch
from torch.utils.data import Dataset

class EmbeddingDataset(Dataset):
    def __init__(self, texts, tokenizer, max_tokens=512):
        """
        Dataset para generar embeddings.

        texts: Lista de textos (ya sea de posts o fact-checks).
        tokenizer: Tokenizador del modelo.
        max_tokens: Máximo número de tokens permitidos por texto.
        """
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_tokens = max_tokens

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        inputs = self.tokenizer(text, truncation=True, max_length=self.max_tokens,
                                padding="max_length", return_tensors="pt")
        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze()
        }

In [8]:
def split_into_chunks(text, max_tokens=512):
    """
    Divide un texto en chunks de hasta max_tokens tokens.
    """
    if not text or not isinstance(text, str):  # Verifica que sea una cadena de texto
        return []
    tokens = tokenizer.encode(text, truncation=True)
    chunks = [tokens[i:i + max_tokens] for i in range(0, len(tokens), max_tokens)]
    return [tokenizer.decode(chunk, skip_special_tokens=True) for chunk in chunks]

# Chunk posts and fact-check texts
posts_mono_sample_1000["chunks"] = posts_mono_sample_1000["ocr_text"].apply(split_into_chunks)

fact_checks_mono_sample_1000["chunks"] = fact_checks_mono_sample_1000["claim_title"].apply(split_into_chunks)

In [9]:
def generate_embeddings(model, dataloader, device):
    model.eval()
    embeddings = []
    with torch.no_grad():  # Desactiva el cálculo de gradientes para eficiencia
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            
            # Obtén las representaciones del modelo (embeddings)
            output = model(input_ids, attention_mask=attention_mask)
            embeddings.append(output)  # Usamos la salida del modelo directamente
    embeddings = torch.cat(embeddings, dim=0)  # Unir todos los embeddings en un solo tensor
    return embeddings

In [10]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def success_at_k(query_embeddings, doc_embeddings, k=10):
    sim_matrix = cosine_similarity(query_embeddings, doc_embeddings)
    top_k_indices = np.argsort(sim_matrix, axis=1)[:, -k:]  # Indices de los 10 más similares
    return top_k_indices


In [11]:
# Preparar textos de posts y fact-checks
posts_texts = posts_mono_sample_1000["chunks"].apply(" ".join).tolist()
fact_checks_texts = fact_checks_mono_sample_1000["chunks"].apply(" ".join).tolist()

# Inicializar los datasets
posts_dataset = EmbeddingDataset(posts_texts, tokenizer)
fact_checks_dataset = EmbeddingDataset(fact_checks_texts, tokenizer)

In [12]:
from torch.utils.data import DataLoader

batch_size = 16
posts_dataloader = DataLoader(posts_dataset, batch_size=batch_size, shuffle=False)
fact_checks_dataloader = DataLoader(fact_checks_dataset, batch_size=batch_size, shuffle=False)

In [13]:
posts_embeddings = generate_embeddings(model_loaded, posts_dataloader, device)
fact_checks_embeddings = generate_embeddings(model_loaded, fact_checks_dataloader, device)

In [14]:
# Asociar embeddings con post_id
posts_embeddings_df = pd.DataFrame(
    posts_embeddings.cpu().numpy(),
    index=posts_mono_sample_1000.index  # Asociar con los nuevos índices
)

# Asociar embeddings con fact_check_id
fact_checks_embeddings_df = pd.DataFrame(
    fact_checks_embeddings.cpu().numpy(),
    index=fact_checks_mono_sample_1000.index
)


In [15]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(posts_embeddings_df, fact_checks_embeddings_df)


In [16]:
similarities

array([[0.9318483 , 0.9139457 , 0.909228  , ..., 0.89053804, 0.9023911 ,
        0.925105  ],
       [0.9029962 , 0.9183223 , 0.9227499 , ..., 0.9042874 , 0.9200471 ,
        0.90308076],
       [0.93127567, 0.9410273 , 0.94506234, ..., 0.93927354, 0.9303847 ,
        0.9426329 ],
       ...,
       [0.9264894 , 0.93210554, 0.93460107, ..., 0.94229364, 0.91962886,
        0.9413129 ],
       [0.9091242 , 0.91025597, 0.9228606 , ..., 0.930479  , 0.9271209 ,
        0.9248395 ],
       [0.8966711 , 0.8822002 , 0.8983729 , ..., 0.8821128 , 0.8896461 ,
        0.8987806 ]], dtype=float32)

In [17]:
# Indices de los 10 más similares por fila
top_10_indices = np.argsort(-similarities, axis=1)[:, :10]

# Crear un DataFrame con las predicciones
predictions = []
for i, indices in enumerate(top_10_indices):
    predictions.append({
        "post_id": posts_mono_sample_1000.index[i],  # ID del post
        "top_k": [{"fact_check_id": fact_checks_mono_sample_1000.index[j]} for j in indices]
    })

predictions_df = pd.DataFrame(predictions)

In [18]:
predictions_df

,post_id,top_k
0,20804,"[{'fact_check_id': 77464}, {'fact_check_id': 9..."
1,10077,"[{'fact_check_id': 45940}, {'fact_check_id': 8..."
2,11419,"[{'fact_check_id': 81755}, {'fact_check_id': 1..."
3,11156,"[{'fact_check_id': 138308}, {'fact_check_id': ..."
4,6132,"[{'fact_check_id': 95232}, {'fact_check_id': 5..."
...,...,...
995,9095,"[{'fact_check_id': 36491}, {'fact_check_id': 9..."
996,19628,"[{'fact_check_id': 131591}, {'fact_check_id': ..."
997,13706,"[{'fact_check_id': 36720}, {'fact_check_id': 1..."
998,14696,"[{'fact_check_id': 40083}, {'fact_check_id': 3..."


In [19]:
def success_at_10(predictions_df, pairs_df):
    """
    Calcula la métrica Success-at-10.

    Args:
        predictions_df (DataFrame): Contiene las predicciones con columnas 'post_id' y 'predicted_fact_check_ids'.
        pairs_df (DataFrame): Contiene las respuestas correctas con columnas 'post_id' y 'fact_check_id'.

    Returns:
        DataFrame: DataFrame con una nueva columna 'success_at_10' indicando si la predicción fue correcta.
    """
    # Crear un diccionario de respuestas correctas: {post_id: set(fact_check_ids)}
    correct_answers = pairs_df.groupby('post_id')['fact_check_id'].apply(set).to_dict()

    # Evaluar cada post_id en las predicciones
    success_scores = []

    for _, row in predictions_df.iterrows():
        post_id = row['post_id']
        predicted_ids = set(int(item['fact_check_id']) for item in row['top_k'])

        # Verificar si alguna predicción es correcta
        success = 1 if post_id in correct_answers and predicted_ids & correct_answers[post_id] else 0
        success_scores.append(success)
    
    # Añadir la columna 'success_at_10' al DataFrame
    predictions_df['success_at_10'] = success_scores

    return predictions_df

In [20]:
# Calcula la métrica Success at 10
results_df = success_at_10(predictions_df, df_fact_check_post_mapping)

# Calcular la métrica global
success_rate = results_df["success_at_10"].mean()
print(f"Success at 10: {success_rate:.2f}")

Success at 10: 0.73


In [21]:
assert predictions_df["post_id"].isin(df_fact_check_post_mapping["post_id"]).all(), "post_id inconsistente"
assert predictions_df["top_k"].apply(lambda x: all(f["fact_check_id"] in df_fact_check_post_mapping["fact_check_id"].values for f in x)).all(), "fact_check_id inconsistente"

In [22]:
# Merge the language information from posts_mono_sample_1000 with results_df
results_with_lang = results_df.merge(
    posts_mono_sample_1000['language'],
    left_on='post_id',
    right_index=True
)

# Calculate Success@10 for each language
language_metrics = results_with_lang.groupby('language').agg({
    'success_at_10': ['mean', 'count']
}).round(3)

# Rename columns for clarity
language_metrics.columns = ['Success@10', 'Count']

# Sort by count descending
language_metrics = language_metrics.sort_values('Count', ascending=False)

print("Success@10 by language:")
print(language_metrics)

Success@10 by language:
          Success@10  Count
language                   
spa            0.735    343
eng            0.704    250
por            0.768    142
fra            0.737     95
msa            0.600     70
ara            0.884     43
deu            0.775     40
tha            0.882     17
